# Sketch2Shape AI — live demo on a Kaggle T4

1. **Settings → Accelerator: GPU T4**, **Internet: on**.
2. Add the dataset **`sketch2shape-src`** (or clone the repo in cell 1).
3. *Run all.* Setup ≈ 7 min + weights ≈ 25 GB; the last cell prints a public `*.gradio.live` link.

Every number in the README was produced by the same scripts (`kaggle/common.sh`, `scripts/setup_backend.sh`).

In [ ]:
import glob, os, shutil, subprocess
# Option A: attached Kaggle dataset; Option B: git clone (edit the URL)
src = glob.glob('/kaggle/input/**/kaggle/tasks.sh', recursive=True)
if src:
    shutil.copytree(os.path.dirname(os.path.dirname(src[0])), '/tmp/s2s/repo', dirs_exist_ok=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/AshmitGarg69/sketch2shape', '/tmp/s2s/repo'], check=True)
%cd /tmp/s2s/repo

In [ ]:
# Setup: 3 environments (built in parallel) + pinned weights (~25 GB). ~10-15 min on a T4.
# Streams live progress (a %%bash cell would only show output at the very end and look "stuck").
import subprocess
script = r"""
source kaggle/common.sh
setup_backend triposr     > $OUT/logs/b_triposr.txt 2>&1 & P1=$!
setup_backend instantmesh > $OUT/logs/b_instantmesh.txt 2>&1 & P2=$!
setup_parent
weights balanced triposr,instantmesh
while kill -0 $P1 2>/dev/null || kill -0 $P2 2>/dev/null; do echo "[$(date +%T)] backend envs still building ($(du -sm envs 2>/dev/null | cut -f1) MB)"; sleep 30; done
wait $P1; wait $P2
cat $OUT/logs/b_*.txt
.venv/bin/python -m sketch2shape.cli doctor
"""
heartbeat = "( while sleep 30; do echo \"[$(date +%T)] working... weights $(du -sm /tmp/s2s/weights 2>/dev/null | cut -f1) MB\"; done ) 2>/dev/null & HB=$!; trap 'pkill -P $HB 2>/dev/null; kill $HB 2>/dev/null' EXIT; "
p = subprocess.Popen(["bash", "-c", heartbeat + script], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end="", flush=True)
assert p.wait() == 0, "setup failed - see the log lines above and /kaggle/working/logs/"

In [ ]:
# Launch the UI with a public link (interactive session: keep this cell running during the demo).
# In a "Save & Run All" (batch) run nobody can see the link, so we generate one test model and stop instead of
# holding the GPU for up to 12 h.
import os, subprocess
env = dict(os.environ, S2S_WEIGHTS='/tmp/s2s/weights', HF_HOME='/tmp/s2s/weights/hf', TORCH_HOME='/tmp/s2s/weights/torch',
           U2NET_HOME='/tmp/s2s/weights/u2net', CUDA_VISIBLE_DEVICES='0', HF_HUB_OFFLINE='1',
           PYTHONUNBUFFERED='1')   # without this, server output (incl. the share link) sits in a pipe buffer and never shows
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'Interactive') == 'Batch':
    r = subprocess.run(['.venv/bin/python', '-m', 'sketch2shape.cli', 'run', '--sketch', 'tests/golden/mug_sketch.jpeg',
                        '--preset', 'balanced', '--seed', '3', '--out', '/kaggle/working/outputs'], env=env, text=True)
    print('batch run finished with exit code', r.returncode, '- outputs in /kaggle/working/outputs')
else:
    proc = subprocess.Popen(['.venv/bin/python', '-m', 'sketch2shape.cli', 'serve', '--share'], env=env,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='')
        if 'gradio.live' in line:
            print('\n>>> Open the link above. Scripted use: sketch2shape run --remote <link> --sketch my.png')